# Audit Risk Screening — Walk-through

This notebook runs the same pipeline as `python run_demo.py`, but pauses at every stage to show what is happening. It is written for an auditor who wants to look under the hood without needing prior machine-learning experience.

The pipeline has four stages:

1. **The data.** A synthetic procurement dataset with 10,000 transactions and 30 quietly planted irregularities.
2. **Transformer fingerprints.** Every transaction is turned into a compact numeric fingerprint — a vector that summarises both its text and its numbers.
3. **Anomaly ranking.** An Isolation Forest looks at all fingerprints together and ranks transactions from most-unusual to most-typical.
4. **Why was it flagged?** SHAP attribution explains which plain-English features pushed a given transaction up the ranking.

Everything below is deterministic: re-run the notebook and you get the same results.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.embed import compute_embeddings
from src.explain import explain_transaction, fit_attribution_model
from src.generate_data import write_dataset
from src.score import get_top_k, recall_at_k, score_transactions
from src.visualize import CATEGORY_COLORS, COLORS

DATA_CSV = Path('data/synthetic_procurement.csv')
EMBEDDING_CACHE = Path('data/embeddings.npy')

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

## Stage 1 — The data

We start by generating the synthetic CSV. It mimics two years of public procurement activity across six ministries, with 200 normal vendors, log-normal amounts, and procurement methods tied to amount tiers. Thirty rows are quietly planted with four kinds of irregularities (threshold avoidance, new-vendor concentration, end-of-period burst, round-number bias).

In a real engagement this CSV would come from the client; here we generate it so the demo is fully reproducible.

In [ ]:
df = write_dataset(DATA_CSV)
print(f'Rows: {len(df):,}')
print(f'Planted irregularities: {(df["_planted_pattern"].astype(str).str.len() > 0).sum()}')
df.head()

In [ ]:
df.describe(include='all').T

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for category, color in CATEGORY_COLORS.items():
    amounts = df.loc[df['category'] == category, 'amount']
    ax.hist(np.log10(amounts), bins=40, alpha=0.5, label=category, color=color)
ax.set_xlabel('log10(amount)')
ax.set_ylabel('Count')
ax.set_title('Amount distribution by category')
ax.legend(fontsize=8, frameon=False)
plt.show()

## Stage 2 — Transformer fingerprints

The text columns (vendor name, category, procurement method, contract description) get glued into one string per transaction and passed through a small pretrained transformer, `sentence-transformers/all-MiniLM-L6-v2`. It returns a 384-dimensional vector that captures the *meaning* of the text — so two transactions described in similar language end up close together, even if the wording is slightly different.

To keep the text from drowning the numeric features inside the Isolation Forest, we then project the text block onto its top principal components (enough to retain 99% of its variance) and concatenate it with the normalised numeric features. The result is a compact fingerprint per transaction. On first run the transformer model downloads (~80 MB). After that the embeddings are cached to disk.

In [ ]:
embeddings = compute_embeddings(df, cache_path=EMBEDDING_CACHE, csv_path=DATA_CSV)
print('Fingerprint matrix shape:', embeddings.shape)

In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
coords = reducer.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(10, 7))
for category, color in CATEGORY_COLORS.items():
    mask = (df['category'] == category).to_numpy()
    ax.scatter(coords[mask, 0], coords[mask, 1], s=10, c=color, alpha=0.5, label=category, linewidths=0)
ax.set_title('Transactions projected to 2D — similar fingerprints sit close')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend(fontsize=8, frameon=False)
plt.show()

## Stage 3 — Anomaly ranking

An Isolation Forest is a simple, fast, unsupervised method for spotting outliers. Intuitively: it builds many random partitions of the data and notes which points get isolated the fastest. Points that get isolated quickly are unusual; points that need many splits to single out are typical.

We fit it on the full fingerprint matrix and add two columns to the DataFrame: `anomaly_score` (normalised 0–1, higher is more anomalous) and `rank` (1 = most anomalous).

In [ ]:
scored = score_transactions(embeddings, df)
top20 = get_top_k(scored, k=20)
top20[['rank', 'transaction_id', 'ministry', 'vendor_name', 'category', 'amount',
       'procurement_method', 'anomaly_score', '_planted_pattern']]

## Stage 4 — Why was it flagged?

A ranked list is only useful if a reviewer can quickly see *why* a transaction landed near the top. We train a small logistic regression on plain-English features (amount, single-source flag, end-of-period flag, vendor history, category/method consistency), then use SHAP to attribute the prediction back to those features for any single row.

Here is the breakdown for **C004** (transaction `T-04004`) — the example referenced in the slide deck.

In [ ]:
model, feature_names, explainer = fit_attribution_model(scored)
attribution = explain_transaction('T-04004', scored, model, explainer)

pd.Series(attribution).sort_values(key=lambda s: s.abs(), ascending=False).to_frame('shap_value')

In [ ]:
normal_id = scored.sort_values('rank', ascending=False).iloc[len(scored) // 2]['transaction_id']
print(f'Contrast example: {normal_id} (a typical transaction near the middle of the ranking)')
pd.Series(explain_transaction(normal_id, scored, model, explainer))\
  .sort_values(key=lambda s: s.abs(), ascending=False).to_frame('shap_value')

## What about the planted irregularities?

The dataset contains 30 known-bad rows across four patterns. A useful screening tool should put most of them near the top of the ranking. We measure this with **recall at top-100** — the fraction of planted rows that show up in the top 100 most-anomalous transactions.

In [ ]:
recall_100 = recall_at_k(scored, k=100)
planted = scored[scored['_planted_pattern'].astype(str).str.len() > 0].copy()
planted['in_top_100'] = planted['rank'] <= 100

print(f'Recall @ top-100: {recall_100:.2f}')
print(f'Planted rows caught: {planted["in_top_100"].sum()} / {len(planted)}')
planted[['transaction_id', '_planted_pattern', 'rank', 'anomaly_score', 'in_top_100']]\
    .sort_values('rank')

## How to adapt this to your data

If you want to run this pipeline on a real procurement extract:

1. Save your CSV with columns matching `data/synthetic_procurement.csv` (or rename inside `src/embed.py` and `src/explain.py`).
2. Drop the `_planted_pattern` column if you don't have ground truth — it is only used for evaluation here.
3. Delete `data/embeddings.npy` so the cache is refreshed.
4. Re-run `python run_demo.py`.

The transformer requires no retraining — it already understands procurement language. The Isolation Forest re-fits automatically on your data. Treat the resulting ranking as a **review queue**, not a verdict: every flagged transaction still needs human judgement before any conclusion is drawn.